# Comparison of Heston (1993), Heston-Nandi (2000) and Christoffersen-Heston-Jacobs (2009) on S&P 500 single-name options

**Setup.** We calibrate three Fourier-inversion option pricing models — Heston
(1993), Heston-Nandi (2000), and Christoffersen-Heston-Jacobs (2009) — daily,
on multiple S&P 500 underlyings, over a 3-year window. Calibration uses
IV-vega weighted RMSE on the filtered option panel. We then perform a
**rolling one-step-ahead out-of-sample** test: parameters fitted at $d$ are
used to price the option chain at $d+1$, $d+2$, $d+5$. Six test months are
split across three volatility regimes (calm / normal / stressed).

**What the notebook shows.** The full analysis loads pre-computed parquet
outputs from `scripts/run_batch.py`. Tables and figures only; the heavy
compute is offline. See `README.md` for the methodology.


## 1. Setup & data overview

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.config import load_config
from src.preprocessing import load_spot_history
from src.analysis import (
    plot_in_sample_heatmap, plot_loss_timeseries,
    plot_oos_by_horizon, plot_oos_by_regime, plot_compute_cost,
    bucket_maturity, bucket_moneyness, classify_regime,
    realized_vol_series, parameter_stability,
    MODEL_NAMES, MODEL_COLORS,
)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

cfg = load_config('../config.yaml')
print(f'Tickers: {cfg.tickers}')
print(f'Period:  {cfg.start_date.date()} -> {cfg.end_date.date()}')
print(f'Models:  {cfg.models}')


In [ ]:
# Load pre-computed results
metrics_path = Path('../results/batch_results.parquet')
params_path  = Path('../results/params_long.parquet')

if not metrics_path.exists():
    raise FileNotFoundError(
        f'{metrics_path} not found. Run: python scripts/run_batch.py first.'
    )

metrics = pd.read_parquet(metrics_path)
metrics['date'] = pd.to_datetime(metrics['date'])

params_long = pd.read_parquet(params_path) if params_path.exists() else pd.DataFrame()
if not params_long.empty:
    params_long['date'] = pd.to_datetime(params_long['date'])

print(f'Loaded {len(metrics)} calibration rows.')
print(f'Coverage: {metrics.ticker.nunique()} tickers x {metrics.model.nunique()} models x {metrics.date.nunique()} dates')
metrics.head()


In [ ]:
# Spot trajectories (normalized) — context
histories = {t: load_spot_history(t, cfg.data_root, cfg.start_date, cfg.end_date)
             for t in cfg.tickers}

fig, ax = plt.subplots(figsize=(11, 5))
for t, h in histories.items():
    if not h.empty:
        norm = h['spot'] / h['spot'].iloc[0]
        ax.plot(h['trade_date'], norm, label=t, linewidth=1.2)
ax.set_ylabel('Spot / Spot[start]'); ax.set_title('Normalized spot trajectories')
ax.legend(loc='upper left', fontsize=9); plt.tight_layout(); plt.show()


In [ ]:
# Data quality stats : option count per (ticker, date)
n_per_day = metrics.groupby(['ticker', 'date'])['n_options'].first().reset_index()
summary = n_per_day.groupby('ticker')['n_options'].agg(['count', 'median', 'min', 'max']).rename(
    columns={'count': 'n_calib_days', 'median': 'n_options_median'})
summary


## 2. Methodology (brief)

| Component | Choice |
|-----------|--------|
| **Filters** | bid>0, mid≥0.10, rel.spread≤25%, DTE∈[7, 365], moneyness∈[0.7, 1.3], vol≥1, mid≥intrinsic |
| **Loss** | IV-vega weighted RMSE (Christoffersen-Jacobs 2004) |
| **Optimization** | L-BFGS-B, log-scale on positive params |
| **HN warmup** | rolling 252-day window for $h_t$ filter |
| **Regimes** | calm / normal / stressed via SPY 21-day realised vol |
| **OOS** | rolling daily : calibrate at $d$, price at $d+\{1,2,5\}$ |


## 3. In-sample fit quality — global

In [ ]:
# Heatmap : median IV-vega RMSE per (model, ticker)
fig = plot_in_sample_heatmap(metrics, metric='in_sample_iv_vega_rmse')
fig.savefig('../results/figures/03_heatmap_in_sample.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Time series of in-sample loss per ticker, 3 models superposed
fig = plot_loss_timeseries(metrics, metric='in_sample_iv_vega_rmse')
fig.savefig('../results/figures/03_timeseries_in_sample.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Summary table : median + IQR per (model, ticker)
recap = metrics.groupby(['model', 'ticker']).agg(
    n=('in_sample_iv_vega_rmse', 'count'),
    median_loss=('in_sample_iv_vega_rmse', 'median'),
    q25=('in_sample_iv_vega_rmse', lambda x: x.quantile(0.25)),
    q75=('in_sample_iv_vega_rmse', lambda x: x.quantile(0.75)),
).round(4)
recap


## 4. In-sample fit — granular (maturity × moneyness)

For each (ticker, model), compute the in-sample RMSE-IV broken down by
maturity bucket (short / medium / long) and moneyness bucket
(otm_put / atm / otm_call). To do this efficiently we need the option-level
prediction errors, not the aggregated loss. We compute them on a representative
sample of calibration dates.

In [ ]:
# This section requires re-pricing options at each (date, model, ticker) and
# recording per-option errors. To keep the notebook fast, we restrict to N
# sample dates per ticker. Set to None to use all dates.
N_SAMPLE_DATES = 6

from src.models import MODEL_REGISTRY, implied_vol
from src.preprocessing import load_chain, FilterConfig, filter_chain, add_implied_vol
from src.calibration import calibrate
from src.calibration.losses import _model_prices
from scripts.run_batch import initial_pricer  # noqa

filt_cfg = FilterConfig(**cfg.filters, r_annual=cfg.calibration['r_annual'])

# Build a sample of dates per ticker (evenly spread across calibrated dates)
sample_records = []
for ticker in cfg.tickers:
    dates_t = sorted(metrics[metrics.ticker == ticker].date.unique())
    if not dates_t: continue
    step = max(1, len(dates_t) // N_SAMPLE_DATES)
    sample_dates = dates_t[::step][:N_SAMPLE_DATES]
    for d in sample_dates:
        sample_records.append((d, ticker))

print(f'Computing granular errors on {len(sample_records)} (date, ticker) combos')


In [ ]:
# This cell is heavy. If you've already run it once, the cached parquet
# `results/granular_errors.parquet` is loaded.
granular_path = Path('../results/granular_errors.parquet')
if granular_path.exists():
    granular = pd.read_parquet(granular_path)
    print(f'Loaded cached granular errors: {len(granular)} option-level rows.')
else:
    rows = []
    for d, ticker in sample_records:
        chain = load_chain(d, ticker, cfg.data_root)
        clean = filter_chain(chain, filt_cfg)
        if len(clean) < 30: continue
        clean = add_implied_vol(clean, cfg.calibration['r_annual'])
        for model_name in cfg.models:
            cls = MODEL_REGISTRY[model_name]
            row_match = metrics[(metrics.date == d) & (metrics.ticker == ticker) & (metrics.model == model_name)]
            if row_match.empty: continue
            # Re-build pricer from the params we already calibrated.
            # NOTE: to keep this lightweight we DON'T re-calibrate; we re-fit a
            # fresh pricer from the recorded params (long-format).
            sub_params = params_long[
                (params_long.date == d) & (params_long.ticker == ticker) & (params_long.model == model_name)
            ].set_index('param_name')['param_value']
            if sub_params.empty: continue
            vec = sub_params.to_numpy() if sub_params.index.is_monotonic_increasing else sub_params.to_numpy()
            pricer = cls.from_calibration_vector(vec, r_f=cfg.calibration['r_annual'])
            mp = _model_prices(pricer, clean)
            for i, (_, opt) in enumerate(clean.iterrows()):
                rows.append({
                    'date': d, 'ticker': ticker, 'model': model_name,
                    'dte': opt['dte'], 'moneyness': opt['moneyness'],
                    'mid': opt['mid'], 'iv_mkt': opt['iv'],
                    'model_price': mp[i],
                    'price_err': mp[i] - opt['mid'],
                })
    granular = pd.DataFrame(rows)
    granular.to_parquet(granular_path, index=False)
    print(f'Computed {len(granular)} option-level rows.')

granular['m_bucket'] = bucket_moneyness(granular['moneyness'])
granular['dte_bucket'] = bucket_maturity(granular['dte'])
granular.head()


In [ ]:
# 3x3 grid per model : median |price_err| by (maturity, moneyness)
fig, axes = plt.subplots(1, len(cfg.models), figsize=(5 * len(cfg.models), 4))
axes = np.atleast_1d(axes)
for ax, m in zip(axes, cfg.models):
    sub = granular[granular.model == m]
    pivot = sub.groupby(['dte_bucket', 'm_bucket'])['price_err'].apply(lambda x: x.abs().median()).unstack()
    pivot = pivot.reindex(index=['short', 'medium', 'long'], columns=['otm_put', 'atm', 'otm_call'])
    im = ax.imshow(pivot.values, cmap='RdYlGn_r', aspect='auto')
    ax.set_xticks(range(3)); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(3)); ax.set_yticklabels(pivot.index)
    for i in range(3):
        for j in range(3):
            v = pivot.values[i, j]
            ax.text(j, i, f'{v:.3f}' if pd.notna(v) else '—', ha='center', va='center')
    ax.set_title(MODEL_NAMES.get(m, m))
fig.suptitle('Median |price error| by (maturity, moneyness)', y=1.03)
plt.tight_layout()
fig.savefig('../results/figures/04_granular_errors.png', dpi=120, bbox_inches='tight')
plt.show()


## 5. Parameter stability over time

In [ ]:
if params_long.empty:
    print('No params_long.parquet — skip')
else:
    stab = parameter_stability(params_long)
    print(f'Stability stats : {len(stab)} (ticker, model, param) groups')
    # Show top-most-unstable params per model
    for m in cfg.models:
        s = stab[stab.model == m].sort_values('cv', ascending=False).head(8)
        print(f'\n--- {m} : params with highest CV ---')
        print(s[['ticker', 'param_name', 'mean', 'std', 'cv', 'autocorr_lag1']].to_string(index=False))


In [ ]:
# Time series of one chosen param per model for AAPL
if not params_long.empty:
    fig, axes = plt.subplots(len(cfg.models), 1, figsize=(11, 3 * len(cfg.models)), sharex=True)
    target_ticker = cfg.tickers[0]
    for ax, m in zip(np.atleast_1d(axes), cfg.models):
        sub = params_long[(params_long.model == m) & (params_long.ticker == target_ticker)]
        for pname, grp in sub.groupby('param_name'):
            grp = grp.sort_values('date')
            ax.plot(grp.date, grp.param_value, '-', label=pname, linewidth=1)
        ax.set_title(f'{MODEL_NAMES.get(m, m)} — {target_ticker}')
        ax.legend(fontsize=7, ncol=3, loc='upper left')
    plt.tight_layout()
    fig.savefig('../results/figures/05_param_stability.png', dpi=120, bbox_inches='tight')
    plt.show()


## 6. Out-of-sample — global view

In [ ]:
# OOS error distribution per horizon
fig = plot_oos_by_horizon(metrics)
if fig is not None:
    fig.savefig('../results/figures/06_oos_by_horizon.png', dpi=120, bbox_inches='tight')
    plt.show()


In [ ]:
# In-sample vs OOS gap (J+1)
gap = metrics.copy()
gap['oos_minus_in'] = gap['oos_iv_vega_rmse_J1'] - gap['in_sample_iv_vega_rmse']

fig, ax = plt.subplots(figsize=(10, 4))
models = sorted(gap.model.unique())
data = [gap[gap.model == m]['oos_minus_in'].dropna().to_numpy() for m in models]
ax.boxplot(data, labels=[MODEL_NAMES.get(m, m) for m in models], showfliers=False)
ax.axhline(0, color='red', linestyle='--', alpha=0.4, label='no gap')
ax.set_ylabel('OOS J+1 loss — In-sample loss')
ax.set_title('Out-of-sample vs in-sample gap (J+1)')
ax.legend()
plt.tight_layout()
fig.savefig('../results/figures/06_oos_gap.png', dpi=120, bbox_inches='tight')
plt.show()


## 7. Out-of-sample by regime (calm / normal / stressed)

In [ ]:
for h in [1, 2, 5]:
    fig = plot_oos_by_regime(metrics, horizon=h)
    if fig is not None:
        fig.savefig(f'../results/figures/07_oos_regime_J{h}.png', dpi=120, bbox_inches='tight')
        plt.show()


In [ ]:
# Numerical recap : median OOS J+1 by (model, regime, ticker)
recap_oos = metrics.groupby(['regime', 'model', 'ticker'])['oos_iv_vega_rmse_J1'].median().unstack('ticker').round(4)
recap_oos


## 8. Computational cost

In [ ]:
fig = plot_compute_cost(metrics)
fig.savefig('../results/figures/08_compute_cost.png', dpi=120, bbox_inches='tight')
plt.show()

print(metrics.groupby('model')['calibration_time_sec'].agg(['median', 'mean', 'std']).round(2))


## 9. Conclusions (to fill once batch has run)

| Use case | Recommended model | Why |
|----------|-------------------|-----|
| Daily risk management on a single ticker | _to fill_ | _from in-sample & OOS results_ |
| Multi-ticker market making | _to fill_ | _from cross-ticker generalisation_ |
| Stress-regime hedging | _to fill_ | _from regime breakdown_ |
| Production speed | _to fill_ | _from compute cost_ |

**Key takeaways** (to fill from the figures above):
- ...

**Limits and extensions:**
- No jumps modelled → short-DTE bias possible.
- 6 test months → limited regime coverage. Extending the OOS calendar is the natural next step.
- Single-ticker calibration → no cross-sectional joint modelling.
